## 1.2 Cleaning and Data Preparation

In this phase we check for null values, duplicates, or structural anomalies to delete it and improve the dataset of `details`, `person_alternative_names`, `person_anime_works`
and `person_details`.
<br>We also identify the number of unique users and the intial distribution of categories.

In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
from lib.dbconnection import create_db_engine
from lib import readSaveCsv as rsc
from lib.clean_utils import clean_studio_format
import pandas as pd

engine = create_db_engine()

details = rsc.import_data(engine, "details")
person_alternate_names = rsc.import_data(engine, "person_alternate_names")
person_anime_works = rsc.import_data(engine, "person_anime_works")
person_details = rsc.import_data(engine, "person_details")

Successo: Dati letti correttamente.
Successo: Dati letti correttamente.
Successo: Dati letti correttamente.
Successo: Dati letti correttamente.


---

### Checking DataFrames info and visualise first rows

In [55]:
details.info()
print('\n')
person_alternate_names.info()
print('\n')
person_anime_works.info()
print('\n')
person_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46124 entries, 0 to 46123
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mal_id           46124 non-null  int64  
 1   title            46124 non-null  object 
 2   title_japanese   45963 non-null  object 
 3   url              46124 non-null  object 
 4   image_url        46124 non-null  object 
 5   type             46124 non-null  object 
 6   status           46124 non-null  object 
 7   score            33437 non-null  float64
 8   scored_by        33437 non-null  float64
 9   start_date       45216 non-null  object 
 10  end_date         20605 non-null  object 
 11  synopsis         39769 non-null  object 
 12  rank             36170 non-null  object 
 13  popularity       46124 non-null  object 
 14  members          46124 non-null  object 
 15  favorites        46124 non-null  object 
 16  genres           46124 non-null  object 
 17  studios     

In [56]:
details

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
0,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503.0,2023-06-11T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,None,None,[],[],[],[]
1,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745.0,2007-01-18T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,None,None,"['Bandai Visual', 'CyberConnect2']",[],[],[]
2,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373.0,2007-12-22T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,None,None,['Bandai Visual'],[],"['Funimation', 'Bandai Entertainment']",[]
3,454,.hack//Gift,.hack//GIFT,https://myanimelist.net/anime/454/hack__Gift,https://cdn.myanimelist.net/images/anime/2/230...,OVA,Finished Airing,6.09,10021.0,2003-11-16T00:00:00+00:00,...,[],Original,R+ - Mild Nudity,1.0,None,None,['CyberConnect2'],[],['Bandai Entertainment'],[]
4,1143,.hack//Intermezzo,.hack//Intermezzo,https://myanimelist.net/anime/1143/hack__Inter...,https://cdn.myanimelist.net/images/anime/1844/...,Special,Finished Airing,6.51,11616.0,2003-03-28T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,None,None,[],[],['Bandai Entertainment'],['Crunchyroll']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46119,59421,Zutaboro Reijou wa Ane no Moto Konyakusha ni D...,ずたぼろ令嬢は姉の元婚約者に溺愛される,https://myanimelist.net/anime/59421/Zutaboro_R...,https://cdn.myanimelist.net/images/anime/1518/...,TV,Finished Airing,7.37,15624.0,2025-07-05T00:00:00+00:00,...,['Josei'],Light novel,PG-13 - Teens 13 or older,12.0,summer,2025.0,"['Studio Pierrot', 'Mainichi Broadcasting Syst...",[],[],"['Crunchyroll', 'Aniplus TV', 'Bahamut Anime C..."
46120,31245,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～,https://myanimelist.net/anime/31245/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/3/821...,Movie,Finished Airing,7.20,104106.0,2016-04-23T00:00:00+00:00,...,[],Music,PG-13 - Teens 13 or older,1.0,None,None,"['Aniplex', 'Dentsu', 'Kadokawa Shoten', 'Movi...",[],['Aniplex of America'],[]
46121,36305,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～ 「金曜日のおはよう」,https://myanimelist.net/anime/36305/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/6/883...,Special,Finished Airing,7.17,10038.0,2016-10-26T00:00:00+00:00,...,[],Music,PG - Children,1.0,None,None,['Aniplex'],[],[],[]
46122,34895,Zutto Suki Datta,ずっと好きだった,https://myanimelist.net/anime/34895/Zutto_Suki...,https://cdn.myanimelist.net/images/anime/1498/...,OVA,Finished Airing,5.68,1887.0,2017-04-21T00:00:00+00:00,...,[],Manga,Rx - Hentai,2.0,None,None,"['Queen Bee', 'Mediabank']",[],[],[]


In [57]:
person_alternate_names

,person_mal_id,alt_name
0,1,Seki Mondoya
1,1,門戸 開
2,1,Monto Hiraku
3,3,雪野五月
4,10,Kevin Hatcher
...,...,...
20421,89567,Sydsnap
20422,89567,Queen of Degeneracy
20423,89826,陳浩
20424,89842,Chidori


In [58]:
person_anime_works

,person_mal_id,position,anime_mal_id
0,1,Theme Song Performance,3080
1,1,Inserted Song Performance,15699
2,1,Theme Song Performance (OP),247
3,1,Theme Song Performance,258
4,1,Theme Song Performance (ED),34825
...,...,...,...
458086,89951,In-Between Animation,11001
458087,89951,Key Animation,55092
458088,89951,"Key Animation (eps 4, 9)",20053
458089,89951,Key Animation,50553


In [59]:
person_details

,person_mal_id,url,website_url,image_url,name,given_name,family_name,birthday,favorites,relevant_location
0,1,https://myanimelist.net/people/1/Tomokazu_Seki,None,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Seki,智一,関,1972-09-08T00:00:00+00:00,6219,"Berlin, Germany"
1,2,https://myanimelist.net/people/2/Tomokazu_Sugita,https://agrs.co.jp/,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Sugita,智和,杉田,1980-10-11T00:00:00+00:00,47666,"Los Angeles, USA"
2,3,https://myanimelist.net/people/3/Satsuki_Yukino,None,https://cdn.myanimelist.net/images/voiceactors...,Satsuki Yukino,さつき,ゆきの,1970-05-25T00:00:00+00:00,1777,"Madrid, Spain"
3,4,https://myanimelist.net/people/4/Aya_Hirano,http://ayahirano.jp/,https://cdn.myanimelist.net/images/voiceactors...,Aya Hirano,綾,平野,1987-10-08T00:00:00+00:00,18374,"Paris, France"
4,5,https://myanimelist.net/people/5/Kenichi_Suzumura,https://intention-k.com,https://cdn.myanimelist.net/images/voiceactors...,Kenichi Suzumura,健一,鈴村,1974-09-12T00:00:00+00:00,5176,"Osaka, Japan"
...,...,...,...,...,...,...,...,...,...,...
76689,90011,https://myanimelist.net/people/90011/Nanako_Ki...,None,https://cdn.myanimelist.net/img/sp/icon/apple-...,Nanako Kishimoto,七子,岸本,None,0,"Mumbai, India"
76690,90012,https://myanimelist.net/people/90012/Pamon,None,https://cdn.myanimelist.net/img/sp/icon/apple-...,Pamon,None,파몬,None,0,"Tokyo, Japan"
76691,90013,https://myanimelist.net/people/90013/Tomoru_Emoto,None,https://cdn.myanimelist.net/img/sp/icon/apple-...,Tomoru Emoto,ともる,柄本,None,0,"Tokyo, Japan"
76692,90014,https://myanimelist.net/people/90014/Hirari,https://hirari.2-d.jp/,https://cdn.myanimelist.net/img/sp/icon/apple-...,Hirari,None,ひらり,None,0,"Paris, France"


### `null` values detection and removal

In [60]:
details = details.dropna(subset=['type', 'status'])
person_details = person_details.dropna(subset=['name'])

### Duplicate rows detection and removal

In [61]:
print(f"Number of rows duplicated (details): {details.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_alternate_names): {person_alternate_names.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_anime_works): {person_anime_works.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_details): {person_details.duplicated().sum()}")

Number of rows duplicated (details): 0

Number of rows duplicated (person_alternate_names): 0

Number of rows duplicated (person_anime_works): 0

Number of rows duplicated (person_details): 0


In [62]:
details = details.drop_duplicates(keep="first")
person_alternate_names = person_alternate_names.drop_duplicates(keep="first")
person_anime_works = person_anime_works.drop_duplicates(keep="first")
person_details = person_details.drop_duplicates(keep="first")
person_anime_works = person_anime_works.dropna(subset=['anime_mal_id'])

In [63]:
print(f"Number of rows duplicated (details): {details.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_alternate_names): {person_alternate_names.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_anime_works): {person_anime_works.duplicated().sum()}\n")
print(f"Number of rows duplicated (person_details): {person_details.duplicated().sum()}")

Number of rows duplicated (details): 0

Number of rows duplicated (person_alternate_names): 0

Number of rows duplicated (person_anime_works): 0

Number of rows duplicated (person_details): 0


### Typo correction

In [64]:
details = details.copy()
person_anime_works = person_anime_works.copy()

details.loc[:, 'genres'] = details['genres'].astype(str).str.replace(r"[\[\]']", "", regex=True)
details['studios'] = details['studios'].apply(clean_studio_format)
details.loc[:, 'mal_id'] = details['mal_id'].astype('int64')
details.loc[:, 'type'] = details['type'].astype('category')
details.loc[:, 'status'] = details['status'].astype('category')

person_anime_works.loc[:, 'position'] = person_anime_works['position'].astype('category')
person_anime_works.loc[:, 'anime_mal_id'] = pd.to_numeric(person_anime_works['anime_mal_id'], errors='coerce')
person_anime_works.loc[:, 'anime_mal_id'] = pd.to_numeric(person_anime_works['anime_mal_id'], errors='coerce')

### Saving in cleaned csv files

In [65]:
rsc.save_data(details, 'details', engine)
rsc.save_data(person_alternate_names, 'person_alternate_names', engine)
rsc.save_data(person_anime_works, 'person_anime_works', engine)
rsc.save_data(person_details, 'person_details', engine)

Successo: 46124 righe caricate nella tabella 'details'.
Successo: 20426 righe caricate nella tabella 'person_alternate_names'.
Successo: 458091 righe caricate nella tabella 'person_anime_works'.
Successo: 76694 righe caricate nella tabella 'person_details'.
